In [ ]:
# Cell 1: Imports and setup
import os
import re
import time
import arxiv
import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from typing import List, Dict, Any, Tuple
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from bertopic import BERTopic
from bertopic.vectorizers import ClassTfidfTransformer
from umap import UMAP

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Ensure directories exist
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)
os.makedirs("data/embeddings", exist_ok=True)
os.makedirs("reports", exist_ok=True)


In [ ]:
# Cell 2: Data Fetching
def fetch_arxiv_papers(query: str, max_results: int = 500, delay: float = 1.0) -> List[Dict[str, Any]]:
    logger.info(f"Fetching up to {max_results} results for query: '{query}'")
    client = arxiv.Client()
    search = arxiv.Search(
        query=query,
        max_results=max_results,
        sort_by=arxiv.SortCriterion.SubmittedDate,
        sort_order=arxiv.SortOrder.Descending,
    )

    records = []
    for i, result in enumerate(client.results(search)):
        records.append({
            "title": result.title,
            "abstract": result.summary,
            "authors": ", ".join(a.name for a in result.authors),
            "year": result.published.year,
            "doi": result.doi or "",
            "arxiv_id": result.entry_id,
        })

        if (i + 1) % 50 == 0:
            logger.info(f"  Fetched {i + 1} records...")
            time.sleep(delay)

    logger.info(f"Done. Total records fetched: {len(records)}")
    return records

records = fetch_arxiv_papers('jo:"Journal of Machine Learning Research"', max_results=500)
raw_df = pd.DataFrame(records)
raw_df.to_csv("data/raw/jmlr_papers.csv", index=False)
print(f"Saved {len(raw_df)} raw records to data/raw/jmlr_papers.csv")


In [ ]:
# Cell 3: Text Preprocessing
def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^a-z0-9\s\.\,\-]', '', text)
    return text.strip()

def process_records(records: List[Dict[str, Any]]) -> pd.DataFrame:
    df = pd.DataFrame(records)
    df = df.dropna(subset=["abstract"])
    df = df[df["abstract"].str.strip() != ""]
    df["clean_abstract"] = [clean_text(t) for t in df["abstract"].tolist()]
    df = df.reset_index(drop=True)
    return df

processed_df = process_records(records)
processed_df = processed_df[~processed_df["clean_abstract"].str.contains(r'\bjo\b', regex=True)]
processed_df.to_csv("data/processed/jmlr_papers_clean.csv", index=False)
print(f"Saved {len(processed_df)} processed records to data/processed/jmlr_papers_clean.csv")


In [ ]:
# Cell 4: Aims and Scope Text Setup
scope_text = """The Journal of Machine Learning Research (JMLR) provides an international 
forum for the electronic and paper publication of high-quality scholarly 
articles in all areas of machine learning. All published papers are freely 
available online. JMLR has a commitment to rigorous yet rapid reviewing.

JMLR seeks previously unpublished papers on machine learning that contain:
- Novel principled machine learning algorithms and their applications
- Analysis of algorithms and the computational complexity of learning
- Studies on the statistical aspects of learning, including consistency and rates of convergence
- Papers that bridge machine learning and statistics, computational complexity, or other fields
- Experimental and theoretical work on neural networks, kernel methods, Bayesian approaches
- Reinforcement learning, unsupervised learning, supervised learning
- Representation learning, deep learning, optimization for machine learning
- Fairness, robustness, and interpretability in machine learning systems
- Applications of machine learning to real-world problems"""

with open("data/aims_and_scope.txt", "w") as f:
    f.write(scope_text)


In [ ]:
# Cell 5: TF-IDF Embedding Generation
def generate_tfidf_embeddings(texts: List[str], max_features: int = 5000) -> np.ndarray:
    vectorizer = TfidfVectorizer(max_features=max_features, stop_words="english", ngram_range=(1, 2))
    return vectorizer.fit_transform(texts).toarray()

abstracts = processed_df["clean_abstract"].fillna("").tolist()
all_texts = [scope_text] + abstracts

print("Generating TF-IDF embeddings...")
tfidf_matrix = generate_tfidf_embeddings(all_texts)
tfidf_scope = tfidf_matrix[0]
tfidf_abstracts = tfidf_matrix[1:]

np.save("data/embeddings/tfidf_scope.npy", tfidf_scope)
np.save("data/embeddings/tfidf_abstracts.npy", tfidf_abstracts)
print(f"TF-IDF shape: {tfidf_abstracts.shape}")


In [ ]:
# Cell 6: SBERT Embedding Generation
def generate_sbert_embeddings(texts: List[str], model_name: str = "all-MiniLM-L6-v2") -> np.ndarray:
    print(f"Loading SBERT model: {model_name} ...")
    model = SentenceTransformer(model_name)
    embeddings = model.encode(texts, show_progress_bar=True, batch_size=32, convert_to_numpy=True)
    return embeddings

print("Generating SBERT embeddings...")
sbert_matrix = generate_sbert_embeddings(all_texts)
sbert_scope = sbert_matrix[0]
sbert_abstracts = sbert_matrix[1:]

np.save("data/embeddings/sbert_scope.npy", sbert_scope)
np.save("data/embeddings/sbert_abstracts.npy", sbert_abstracts)
print(f"SBERT shape: {sbert_abstracts.shape}")


In [ ]:
# Cell 7: Alignment Evaluation & Outlier Detection
def compute_alignment_scores(scope_vector: np.ndarray, abstract_matrix: np.ndarray) -> np.ndarray:
    return cosine_similarity(scope_vector.reshape(1, -1), abstract_matrix).flatten()

def detect_outliers(df: pd.DataFrame, score_col: str = "sbert_alignment_score", threshold: float = 2.0) -> pd.DataFrame:
    mean = df[score_col].mean()
    std = df[score_col].std()
    
    df = df.copy()
    df["score_mean"] = mean
    df["score_std"] = std
    df["outlier_type"] = None
    df.loc[df[score_col] > mean + threshold * std, "outlier_type"] = "high"
    df.loc[df[score_col] < mean - threshold * std, "outlier_type"] = "low"
    return df

df_scored = processed_df.copy()

print("Computing TF-IDF alignment scores...")
df_scored["tfidf_alignment_score"] = compute_alignment_scores(tfidf_scope, tfidf_abstracts)

print("Computing SBERT alignment scores...")
df_scored["sbert_alignment_score"] = compute_alignment_scores(sbert_scope, sbert_abstracts)

print("Detecting outliers (±2 std dev threshold)...")
df_scored = detect_outliers(df_scored, score_col="sbert_alignment_score", threshold=2.0)

mean = df_scored["score_mean"].iloc[0]
std = df_scored["score_std"].iloc[0]
n_high = (df_scored["outlier_type"] == "high").sum()
n_low = (df_scored["outlier_type"] == "low").sum()

print(f"Corpus mean:  {mean:.4f}")
print(f"Corpus std:   {std:.4f}")
print(f"Upper bound:  {mean + 2 * std:.4f}  →  {n_high} high outliers")
print(f"Lower bound:  {mean - 2 * std:.4f}  →  {n_low} low outliers")

df_scored.to_csv("data/processed/jmlr_papers_scored.csv", index=False)


In [ ]:
# Cell 8: Topic Modeling (BERTopic)
def run_topic_modeling(abstracts: List[str], embeddings: np.ndarray, n_topics: int = 15, min_topic_size: int = 10) -> Tuple[pd.DataFrame, Dict[int, str], pd.DataFrame]:
    print(f"Fitting BERTopic on {len(abstracts)} abstracts...")
    vectorizer = CountVectorizer(stop_words="english", ngram_range=(1, 2), min_df=2)
    ctfidf = ClassTfidfTransformer(reduce_frequent_words=True)
    
    model = BERTopic(
        nr_topics=n_topics,
        min_topic_size=min_topic_size,
        vectorizer_model=vectorizer,
        ctfidf_model=ctfidf,
        calculate_probabilities=False,
        verbose=True,
    )
    
    topics, _ = model.fit_transform(abstracts, embeddings=embeddings)
    topic_info = model.get_topic_info()
    
    labels = {}
    for tid in topic_info["Topic"].tolist():
        if tid == -1:
            labels[tid] = "outlier / noise"
            continue
        words = model.get_topic(tid)
        if words:
            labels[tid] = ", ".join([w for w, _ in words[:3]])
            
    return topic_info, labels, pd.DataFrame({"topic_id": topics})

topic_info, topic_labels, df_topics_raw = run_topic_modeling(
    abstracts, sbert_abstracts, n_topics=20, min_topic_size=8
)

df_topics = df_scored.copy()
df_topics["topic_id"] = df_topics_raw["topic_id"]
df_topics["topic_label"] = df_topics["topic_id"].map(topic_labels)
df_topics.to_csv("data/processed/jmlr_papers_topics.csv", index=False)

summary = (
    df_topics[df_topics["topic_id"] != -1]
    .groupby(["topic_id", "topic_label"])
    .agg(
        n_papers=("title", "count"),
        mean_alignment=("sbert_alignment_score", "mean"),
        std_alignment=("sbert_alignment_score", "std"),
    )
    .reset_index()
    .sort_values("mean_alignment", ascending=False)
)
summary.to_csv("data/processed/topic_alignment_summary.csv", index=False)

print("\nTopic alignment summary:")
print(summary.to_string(index=False))


In [ ]:
# Cell 9: UMAP Projection
def run_umap_projection(embeddings: np.ndarray, n_neighbors: int = 15, min_dist: float = 0.1) -> np.ndarray:
    print(f"Running UMAP on {embeddings.shape[0]} vectors of dim {embeddings.shape[1]}...")
    reducer = UMAP(n_components=2, n_neighbors=n_neighbors, min_dist=min_dist, metric="cosine", random_state=42)
    return reducer.fit_transform(embeddings)

embedding_2d = run_umap_projection(sbert_abstracts)

df_umap = df_topics.copy()
df_umap["umap_x"] = embedding_2d[:, 0]
df_umap["umap_y"] = embedding_2d[:, 1]
df_umap.to_csv("data/processed/jmlr_papers_umap.csv", index=False)

print(f"UMAP projection complete. Ranges: x=[{df_umap['umap_x'].min():.2f}, {df_umap['umap_x'].max():.2f}], y=[{df_umap['umap_y'].min():.2f}, {df_umap['umap_y'].max():.2f}]")


In [ ]:
# Cell 10: Plotting Functions
def plot_score_distribution(df: pd.DataFrame) -> None:
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.histplot(data=df, x="sbert_alignment_score", bins=40, kde=True, ax=ax, color="#4C72B0")
    ax.set_title("JMLR — Distribution of SBERT alignment scores", fontsize=13)
    ax.set_xlabel("Cosine similarity to Aims & Scope")
    ax.set_ylabel("Number of papers")
    plt.tight_layout()
    plt.show()

def plot_temporal_drift(df: pd.DataFrame) -> None:
    yearly = df.groupby("year")["sbert_alignment_score"].agg(["mean", "count"]).reset_index()
    fig, ax1 = plt.subplots(figsize=(10, 5))
    sns.regplot(data=df, x="year", y="sbert_alignment_score", scatter=False,
                ci=95, line_kws={"color": "red", "linestyle": "--"}, ax=ax1)
    sns.lineplot(data=yearly, x="year", y="mean", marker="o", linewidth=2, color="#4C72B0", ax=ax1)
    ax1.set_title("JMLR — Mean thematic alignment score over time", fontsize=13)
    ax1.set_xlabel("Publication year")
    ax1.set_ylabel("Mean SBERT alignment score")
    ax1.set_xticks(yearly["year"].unique())
    plt.tight_layout()
    plt.show()

def plot_outliers(df: pd.DataFrame, n: int = 10) -> None:
    top = df.nlargest(n, "sbert_alignment_score")
    bottom = df.nsmallest(n, "sbert_alignment_score")
    concat_df = pd.concat([top, bottom]).sort_values("sbert_alignment_score")
    colors = ["#d62728" if score < df["sbert_alignment_score"].mean() else "#2ca02c"
              for score in concat_df["sbert_alignment_score"]]
    fig, ax = plt.subplots(figsize=(12, 8))
    bars = ax.barh(concat_df["title"].str.slice(0, 70) + "...",
                   concat_df["sbert_alignment_score"], color=colors)
    ax.axvline(df["sbert_alignment_score"].mean(), color="black", linestyle="--", alpha=0.6, label="Corpus Mean")
    ax.set_title(f"JMLR — Highest and lowest aligned papers (Top/Bottom {n})", fontsize=13)
    ax.set_xlabel("SBERT alignment score")
    ax.legend()
    plt.tight_layout()
    plt.show()

def plot_topic_distribution(df: pd.DataFrame) -> None:
    summary = (df[df["topic_id"] != -1].groupby(["topic_id", "topic_label"])
               .agg(n_papers=("title", "count"), mean_alignment=("sbert_alignment_score", "mean"))
               .reset_index().sort_values("n_papers", ascending=True))
    fig, ax = plt.subplots(figsize=(12, 8))
    norm = plt.Normalize(summary["mean_alignment"].min(), summary["mean_alignment"].max())
    colors = plt.cm.RdYlGn(norm(summary["mean_alignment"]))
    ax.barh(summary["topic_label"], summary["n_papers"], color=colors, edgecolor="white")
    ax.set_xlabel("Number of papers")
    ax.set_title("JMLR — Papers per topic (color = mean alignment score)", fontsize=13)
    sm = plt.cm.ScalarMappable(cmap="RdYlGn", norm=norm)
    plt.colorbar(sm, ax=ax, label="Mean SBERT alignment score", shrink=0.6)
    plt.tight_layout()
    plt.show()

def plot_topic_drift(df: pd.DataFrame, top_n_topics: int = 6) -> None:
    top_topics = (df[df["topic_id"] != -1].groupby("topic_label")["title"]
                  .count().nlargest(top_n_topics).index.tolist())
    yearly = (df[df["topic_label"].isin(top_topics)].groupby(["year", "topic_label"])["title"]
              .count().reset_index(name="count"))
    totals = df.groupby("year")["title"].count().reset_index(name="total")
    yearly = yearly.merge(totals, on="year")
    yearly["share"] = yearly["count"] / yearly["total"]
    fig, ax = plt.subplots(figsize=(12, 5))
    for topic, group in yearly.groupby("topic_label"):
        ax.plot(group["year"], group["share"], marker="o", linewidth=2, label=topic)
    ax.set_title(f"JMLR — Publication share of top {top_n_topics} topics over time", fontsize=13)
    ax.set_xlabel("Year")
    ax.set_ylabel("Share of papers")
    ax.yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1, decimals=0))
    ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=9)
    plt.tight_layout()
    plt.show()

def plot_topic_year_heatmap(df: pd.DataFrame) -> None:
    pivot = (df[df["topic_id"] != -1].groupby(["topic_label", "year"])["sbert_alignment_score"]
             .mean().unstack(level="year"))
    fig, ax = plt.subplots(figsize=(13, 6))
    sns.heatmap(pivot, ax=ax, cmap="RdYlGn", annot=True, fmt=".2f", linewidths=0.4,
                linecolor="white", cbar_kws={"label": "Mean SBERT alignment score", "shrink": 0.7})
    ax.set_title("JMLR — Mean alignment score by topic and year", fontsize=13)
    ax.set_xlabel("Year")
    ax.set_ylabel("")
    plt.tight_layout()
    plt.show()

def plot_outlier_distribution(df: pd.DataFrame) -> None:
    mean, std = df["score_mean"].iloc[0], df["score_std"].iloc[0]
    upper, lower = mean + 2 * std, mean - 2 * std
    fig, ax = plt.subplots(figsize=(11, 5))
    ax.hist(df["sbert_alignment_score"], bins=40, color="#4C72B0", alpha=0.7, edgecolor="white", label="All papers")
    high = df[df["outlier_type"] == "high"]["sbert_alignment_score"]
    low = df[df["outlier_type"] == "low"]["sbert_alignment_score"]
    if not high.empty:
        ax.hist(high, bins=40, color="#2ca02c", alpha=0.9, edgecolor="white", label=f"High outliers (n={len(high)})")
    if not low.empty:
        ax.hist(low, bins=40, color="#d62728", alpha=0.9, edgecolor="white", label=f"Low outliers (n={len(low)})")
    ax.axvline(mean, color="black", linestyle="-", linewidth=1.5, label=f"Mean: {mean:.3f}")
    ax.axvline(upper, color="green", linestyle="--", linewidth=1.5, label=f"+2σ: {upper:.3f}")
    ax.axvline(lower, color="red", linestyle="--", linewidth=1.5, label=f"−2σ: {lower:.3f}")
    ax.fill_betweenx([0, ax.get_ylim()[1] if ax.get_ylim()[1] > 0 else 60], lower, upper, alpha=0.05, color="gray", label="±2σ band")
    ax.set_xlabel("SBERT alignment score")
    ax.set_ylabel("Number of papers")
    ax.set_title("JMLR — Alignment score distribution with outlier thresholds (±2σ)", fontsize=13)
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.show()

def plot_umap_by_topic(df: pd.DataFrame) -> None:
    fig, ax = plt.subplots(figsize=(13, 8))
    noise = df[df["topic_id"] == -1]
    ax.scatter(noise["umap_x"], noise["umap_y"], c="lightgrey", s=18, alpha=0.4, linewidths=0, label="noise (topic −1)")
    topics = df[df["topic_id"] != -1]
    palette = sns.color_palette("tab10", n_colors=topics["topic_id"].nunique())
    for i, (tid, group) in enumerate(topics.groupby("topic_id")):
        label = group["topic_label"].iloc[0]
        short = label[:35] + "…" if len(label) > 35 else label
        ax.scatter(group["umap_x"], group["umap_y"], color=palette[i], s=28, alpha=0.8, linewidths=0, label=f"{tid}: {short}")
    ax.set_title("JMLR — UMAP projection colored by topic", fontsize=14)
    ax.set_xlabel("UMAP dimension 1")
    ax.set_ylabel("UMAP dimension 2")
    ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8, frameon=False)
    plt.tight_layout()
    plt.show()

def plot_umap_by_alignment(df: pd.DataFrame) -> None:
    fig, ax = plt.subplots(figsize=(11, 7))
    sc = ax.scatter(df["umap_x"], df["umap_y"], c=df["sbert_alignment_score"], cmap="RdYlGn", s=28, alpha=0.85, linewidths=0,
                    vmin=df["sbert_alignment_score"].quantile(0.05), vmax=df["sbert_alignment_score"].quantile(0.95))
    outliers_low = df[df["outlier_type"] == "low"]
    outliers_high = df[df["outlier_type"] == "high"]
    if not outliers_low.empty:
        ax.scatter(outliers_low["umap_x"], outliers_low["umap_y"], edgecolors="red", facecolors="none", s=80, linewidths=1.5, label="Low outlier (−2σ)")
    if not outliers_high.empty:
        ax.scatter(outliers_high["umap_x"], outliers_high["umap_y"], edgecolors="green", facecolors="none", s=80, linewidths=1.5, label="High outlier (+2σ)")
    plt.colorbar(sc, ax=ax, label="SBERT alignment score", shrink=0.7)
    ax.set_title("JMLR — UMAP projection colored by alignment score", fontsize=14)
    ax.set_xlabel("UMAP dimension 1")
    ax.set_ylabel("UMAP dimension 2")
    if not outliers_low.empty or not outliers_high.empty:
        ax.legend(fontsize=9, frameon=False)
    plt.tight_layout()
    plt.show()

def plot_umap_by_year(df: pd.DataFrame) -> None:
    fig, ax = plt.subplots(figsize=(11, 7))
    years = sorted(df["year"].unique())
    palette = sns.color_palette("coolwarm", n_colors=len(years))
    year_color = {y: palette[i] for i, y in enumerate(years)}
    for year, group in df.groupby("year"):
        ax.scatter(group["umap_x"], group["umap_y"], color=year_color[year], s=28, alpha=0.8, linewidths=0, label=str(year))
    ax.set_title("JMLR — UMAP projection colored by publication year", fontsize=14)
    ax.set_xlabel("UMAP dimension 1")
    ax.set_ylabel("UMAP dimension 2")
    ax.legend(title="Year", bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=9, frameon=False)
    plt.tight_layout()
    plt.show()


In [ ]:
# Cell 11: Run Visualizations
plot_score_distribution(df_umap)
plot_temporal_drift(df_umap)
plot_outliers(df_umap, n=10)
plot_topic_distribution(df_umap)
plot_topic_drift(df_umap)
plot_topic_year_heatmap(df_umap)
plot_outlier_distribution(df_umap)
plot_umap_by_topic(df_umap)
plot_umap_by_alignment(df_umap)
plot_umap_by_year(df_umap)
